# ECE457B A3 - Q3.5: ECE457B-GPT

Enter your name and Waterloo ID in the first code cell. Place `frankenstein.txt` beside this notebook. Fill in the incomplete code, run the notebook, and include your settings, initial and final training and validation losses, and a generated text sample in the exported PDF. You may reduce training iterations or model size to fit your available compute. Upload that PDF to Crowdmark and the `.ipynb` source to the A3 LEARN Dropbox.

This task uses **character** tokens; Q3.1 uses word tokens.
Adapt the attention concepts rather than copying token IDs directly.

In [1]:
# Name: Michael McPhee
# Waterloo ID: 21021347


## Setup

In [1]:
# Run this setup code:

# init libraries
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64
block_size = 256
max_iters = 3000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 256
n_head = 4
n_layer = 4
dropout = 0.2

torch.manual_seed(123123)
torch.set_printoptions(linewidth=80)

print(f'device: {device}')
print(f'batch_size={batch_size}, block_size={block_size}, max_iters={max_iters},\n'
      f'eval_interval={eval_interval}')
print(f'learning_rate={learning_rate}, eval_iters={eval_iters},\n'
      f'n_embd={n_embd}, n_head={n_head}, n_layer={n_layer},\n'
      f'dropout={dropout}')


device: cuda
batch_size=64, block_size=256, max_iters=3000, eval_interval=500
learning_rate=0.0003, eval_iters=50, n_embd=256, n_head=4, n_layer=4, dropout=0.2


## Load and tokenize the text

In [3]:
# Here we process the .txt file and create our training and validation set

# Place frankenstein.txt alongside the notebook before running it.
with open('frankenstein.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Q3.5: character tokenization
# Adapt the tokenization ideas from Q3.1. Here, tokens are characters.
chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
def encode(s):
    return [stoi[ch] for ch in s]

itos = {i: ch for i, ch in enumerate(chars)}
def decode(i):
    return ''.join(itos[idx] for idx in i)

# Now we split our data into a 80/20 train and val splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]
print(f'characters: {len(text)}, vocabulary: {vocab_size},\n'
      f'train tokens: {len(train_data)}, validation tokens: {len(val_data)}')


characters: 438841, vocabulary: 93, train tokens: 351072, validation tokens: 87769


## Implement attention heads

In [4]:
# Define one self-attention head, then combine multiple heads.

# Q3.5: causal self-attention
# Adapt the attention ideas from Q3.2 and Q3.3.
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wx = q @ k.transpose(-2, -1) / (k.size(-1) ** 0.5)
        wx = wx.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        alphaxi = self.dropout(F.softmax(wx, dim=-1))
        C = alphaxi @ v

        return C

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


## Feed-forward network and attention block

In [5]:
# Now we create  our Feed Forward Network and Attention Block

class FeedFoward(nn.Module):
    """ Here we define a feed forward network with the following structure """
    # for n_embd: embedding dimension
    # a linear layer (n_embd x 4 * n_embd)
    # a ReLU layer
    # another linear layer (n_embd x 4 * n_embd)
    # a Dropout layer

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class AttentionBlock(nn.Module):
    """ Here we create our Attention Block with the following structure """
    # for n_embd: embedding dim, n_head: number of heads
    # layer norm layer
    # multi-head attention layer
    # layer norm layer
    # feed forward layer

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


## Assemble the model

In [6]:
# Now we put everything together for our final model

class ECE457BGPT(nn.Module):
    # a token embedding layer of shape: vocab_size x embedding_dim
    # a position embedding layer of shape: block_size x embedding_dim
    # a sequential multi_block layer that creates n AttentionBlocks, each of shape: n_embd x n_head
    # a LayerNorm layer of size n_embd
    # a final Linear layer of shape: n_embd x vocab_size
    def __init__(self):
        super().__init__()
        ## create the layers that the model will use given the structure above
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[AttentionBlock(n_embd, n_head=n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Initialize weights to improve training speed.
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            ## compute the cross-entropy loss for the logits and the target values
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] ## crop idx to the last block_size tokens
            logits, loss = self(idx_cond) ## get the predictions
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1) ## apply softmax to get probabilities
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


## Train and generate text

In [7]:
# Now we optimize our model and evaluate it's performance
model = ECE457BGPT()
m = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate
)

# helper function that creates a small batch of the data to validate the models performance
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


## for loop for estimating the train and val loss.
# prints the average train and val losses for the previous steps at every timestep of 500
# Uses the helper function to sample a batch of training data, evaluates the loss on it and uses the optimizer to backward prop through the model

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

initial_losses = None
final_losses = None
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        if initial_losses is None:
            initial_losses = {split: losses[split].item() for split in losses}
        if iter == max_iters - 1:
            final_losses = {split: losses[split].item() for split in losses}
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"initial losses: train={initial_losses['train']:.4f}, val={initial_losses['val']:.4f}")
print(f"final losses: train={final_losses['train']:.4f}, val={final_losses['val']:.4f}")

#You can finally test your model's ability to generate text using this line of code!
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


step 0: train loss 4.5526, val loss 4.5508


step 500: train loss 1.7674, val loss 1.9220


step 1000: train loss 1.3576, val loss 1.6144


step 1500: train loss 1.1895, val loss 1.5262


step 2000: train loss 1.0959, val loss 1.4984


step 2500: train loss 1.0270, val loss 1.5098


step 2999: train loss 0.9498, val loss 1.5135
initial losses: train=4.5526, val=4.5508
final losses: train=0.9498, val=1.5135



lipen can being with peace and savitions. The empt of
Grmef’s Mreak, walls and by God’s solitary. We
continued more the wretchle day lesson of scendly. TI
ideas allowed my tale. Oh! I gazed my eyesterin spirit. But I was drince near his inece, so
deling to seek pen, accorning that injure you had been for
this? Nay follow me, inte once my freal day, when other you nor only
repayer esaw fringed me where words and some heaven, mad all, my men broatoring, despair
among had obligen. One of happy, my 
